# Sentiment Analysis using LLMs - Part 1

This example provides a very quick overview of LangDB functionality.

## Step 1: Prepping the tables
- Create products data
- Create customer_reviews data

In [1]:
CREATE TABLE products (
    product_id UInt32,
    product_name String
) ENGINE = MergeTree
ORDER BY product_id;

In [2]:
CREATE TABLE customer_reviews (
    review_id UInt32,
    product_id UInt32,
    customer_id UInt32,
    review_date Date,
    review_text String,
    rating UInt8,
    location String
) ENGINE = MergeTree
ORDER BY review_id;

In [3]:
INSERT INTO products (product_id, product_name) VALUES 
(1, 'Product A'),
(2, 'Product B'),
(3, 'Product C');

In [4]:
INSERT INTO customer_reviews (product_id, customer_id, review_date, review_text, rating, location) VALUES 
(1, 101, '2024-05-01', 'This product is fantastic! Highly recommend it.', 5, 'New York, NY'),
(2, 102, '2024-05-02', 'Not worth the money. Poor quality.', 2, 'Los Angeles, CA'),
(1, 103, '2024-05-03', 'Good value for the price. Satisfied with my purchase.', 4, 'Chicago, IL'),
(2, 104, '2024-05-04', 'Terrible customer service and the product broke after a week.', 1, 'Houston, TX'),
(3, 105, '2024-05-05', 'Excellent product! Will buy again.', 5, 'Phoenix, AZ');


In [8]:
select product_id, count(product_id) as no_reviews from products as p join customer_reviews as cr on cr.product_id = p.product_id  
group by product_id

,product_id,no_reviews
0,3,1
1,2,2
2,1,2


## Step 2: Create Model and Prompt

In [16]:
CREATE PROMPT customer_review_pt (
  system "You are an AI assistant. Look at the provided review and return average sentiment score for the review.
    Strictly only return a number between 1 and 5 as a value without any description.
    ",
  human "{{review}}"
);

In [ ]:
CREATE PROVIDER open_ai_provider
ENGINE = OpenAI(
	api_key='sk-proj-xxx',
	model_name = 'gpt-3-turbo'
);

In [17]:
CREATE MODEL review(
    review
) USING open_ai_provider()
PROMPT customer_review_pt;

That's all you needed you can run your query as if you would use any function. 

Let's run it on one row and test the result

In [22]:
select toInt32(review('This product is fantastic! Highly recommend it.', 5, 'New York, NY')) as product_review

,product_review
0,5


In [21]:
SELECT 
    p.product_id, 
    p.product_name,  
    AVG(toInt32(review(cr.review_text))) as avg_sentiment_score
FROM  
    customer_reviews cr   
JOIN 
    products p 
ON 
    cr.product_id = p.product_id   
GROUP BY 
    p.product_id, p.product_name   
ORDER BY 
    avg_sentiment_score DESC;

,p.product_id,product_name,avg_sentiment_score
0,3,Product C,5.0
1,1,Product A,4.5
2,2,Product B,1.5


## Why is this a big deal ?

Well you might be thinking sentiment analysis has been around for ages and LLMs can be extremely slow for the purpose. So let's make this an interesting example. So far even though it worked great, we are only passing one row at a time which will translate into many number of API calls and too much slowness.

**New Approach**
- Send batch of 10 rows at a time leveraging `pretty_print()` function.
- Store intermediate results and create an `Agent` for accessing averages so LLM can have a history of deviations
- Instruct LLM to lag out specific extreme reviews that deviate too much for a product in a response.
- Meanwhile also calculate the review.
- Lets also generate reasonable sized data to get some interesting insights.

Models will keep getting better in the AI world and LangDB gives you a way to dynamically combine RAG and models in a easy way to create many exciting applications.

As this post has become already too long, lets continue this in a separate post (Part 2). 

